# py-mea-axion — LGI2 KD worked example

This notebook demonstrates the full `py-mea-axion` analysis pipeline using a
real Axion Biosystems recording from an LGI2 knockdown experiment.

**Experimental design (Plate 2, DIV 28)**

| Wells | Condition | Batch |
|---|---|---|
| A1, B1, C1, D1 | SCRM (scramble control) | 3 |
| A2, B2, C2, D2 | LGI2 KD4 | 3 |
| A3, B3, C3, D3 | LGI2 KD5 | 3 |

**Pipeline steps covered**
1. Load `.spk` binary file and define plate map
2. Compute per-electrode spike metrics
3. Detect single-electrode bursts
4. Detect network bursts
5. Compute pairwise STTC synchrony
6. Visualise results
7. Statistical comparison across conditions

## 0. Installation

```bash
pip install py-mea-axion
```

Or from source:

```bash
git clone https://github.com/Molecularbiologyworld/py-mea-axion.git
cd py-mea-axion
pip install -e .
```

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

from py_mea_axion.pipeline import MEAExperiment

%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 100   # screen display; savefig uses 600 DPI

## 1. Define the plate map

The plate map links each well to its experimental condition. It can be
loaded from a CSV file (`pd.read_csv('plate_map.csv')`) or defined
inline as shown here.

In [ ]:
metadata = pd.DataFrame({
    'well_id':      ['A1','B1','C1','D1',
                     'A2','B2','C2','D2',
                     'A3','B3','C3','D3'],
    'condition':    ['SCRM']*4 + ['KD4']*4 + ['KD5']*4,
    'batch':        [3]*12,
    'DIV':          [28]*12,
    'replicate_id': ['Plate2_A1','Plate2_B1','Plate2_C1','Plate2_D1',
                     'Plate2_A2','Plate2_B2','Plate2_C2','Plate2_D2',
                     'Plate2_A3','Plate2_B3','Plate2_C3','Plate2_D3'],
})

metadata

## 2. Load the recording and run the pipeline

`MEAExperiment` reads the `.spk` binary file and runs the complete
analysis in one call.

**Burst detection parameters used here:**
- Algorithm: ISI threshold
- Max within-burst ISI: 100 ms
- Minimum spikes per burst: 50

**Network burst parameters:**
- Minimum electrode participation: 35 %

> **Note:** This recording lacks a `BlockVectorHeader`, which is common
> in some Axion firmware versions. We supply `fs_override=12500` to set
> the sampling frequency manually.

In [ ]:
SPK_FILE = Path('../LGI2 KD data/20251004_LGI2 KD_Plate 2_D28N(000).spk')

exp = MEAExperiment(
    SPK_FILE,
    metadata=metadata,
    fs_override=12500,
    burst_kwargs={
        'algorithm':  'isi_threshold',
        'max_isi_s':  0.1,
        'min_spikes': 50,
    },
    network_kwargs={
        'participation_threshold': 0.35,
    },
).run()

print(f'Recording duration : {exp.total_time_s:.0f} s')
print(f'Wells loaded       : {exp.wells}')

## 3. Per-electrode spike metrics

`spike_metrics` contains one row per electrode with mean firing rate
(MFR), inter-spike interval (ISI) statistics, and an active-electrode
flag (MFR ≥ 0.1 Hz by default).

In [ ]:
exp.spike_metrics.head(10)

In [ ]:
# Summary counts per well
(exp.spike_metrics
   .groupby('well_id')
   .agg(n_electrodes=('electrode_id','count'),
        n_active=('is_active','sum'),
        mean_mfr=('mfr_hz','mean'))
   .round(3))

## 4. Per-well summary

`well_summary` aggregates spike, burst, and synchrony metrics to the
well level and merges in the plate-map metadata.

In [ ]:
# Filter to the 12 seeded wells and show key metrics
ws = exp.joined_summary()
ws = ws[ws['condition'].notna()].reset_index(drop=True)

ws[['well_id','condition','n_active','mean_mfr_active_hz',
    'burst_rate_hz','n_network_bursts','mean_sttc']].round(3)

## 5. Burst table

`burst_table` contains one row per detected burst across all electrodes
and wells.

In [ ]:
bt = exp.burst_table
print(f'Total bursts detected: {len(bt)}')
bt.head(10)

## 6. Visualisation

### 6.1 Electrode MFR heatmap grid

Each panel shows the 4 × 4 electrode grid of one well, coloured by
mean firing rate. All panels share the same colour scale.

In [ ]:
from py_mea_axion.viz.heatmap import plot_electrode_heatmap

ACTIVE_WELLS = ['A1','B1','C1','D1','A2','B2','C2','D2','A3','B3','C3','D3']
COND = {w: c for w, c in zip(ACTIVE_WELLS, ['SCRM']*4+['KD4']*4+['KD5']*4)}

sm = exp.spike_metrics
active_mfr = sm.loc[sm['is_active'], 'mfr_hz']
vmax = float(np.percentile(active_mfr, 95)) if len(active_mfr) else 1.0

fig, axes = plt.subplots(4, 3, figsize=(11, 13))
for idx, well in enumerate(ACTIVE_WELLS):
    ax   = axes[idx % 4, idx // 4]
    rows = sm[sm['well_id'] == well]
    vals = dict(zip(rows['electrode_id'], rows['mfr_hz']))
    plot_electrode_heatmap(vals, well_id=well, metric_name='MFR (Hz)',
                           vmin=0, vmax=vmax, ax=ax,
                           title=f"{well}  [{COND[well]}]")
for ax in axes.flat[12:]:
    ax.set_visible(False)
fig.suptitle('Plate 2 · DIV 28 · Mean Firing Rate per electrode', fontsize=13)
fig.tight_layout()
plt.show()

### 6.2 Burst raster

Spike times (vertical ticks) and burst periods (orange shading) for
one representative well per condition.

In [ ]:
for well, cond in [('A1','SCRM'), ('A2','KD4'), ('A3','KD5')]:
    n_nb = len(exp.network_bursts.get(well, []))
    fig  = exp.plot_raster(
        well,
        title=f"{well}  [{cond}]  ·  DIV 28  ·  {n_nb} network bursts",
        figsize=(10, 4),
    )
    plt.show()
    plt.close(fig)

### 6.3 ISI distribution

Log-scale inter-spike interval histogram for the most active electrode
in each condition. A bimodal distribution (short ISIs from within-burst
spiking, long ISIs from between-burst silence) is characteristic of
bursting activity.

In [ ]:
from py_mea_axion.viz.burst_charts import plot_isi_histogram

PALETTE = {'SCRM':'#0072B2', 'KD4':'#D55E00', 'KD5':'#009E73'}

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, (well, cond) in zip(axes, [('A1','SCRM'), ('A2','KD4'), ('A3','KD5')]):
    ws   = exp.well_spikes(well)
    best = (sm[sm['well_id'] == well]
            .sort_values('mfr_hz', ascending=False)
            .iloc[0]['electrode_id'])
    plot_isi_histogram(ws[best], electrode_id=f"{best}  [{cond}]",
                       log_x=True, color=PALETTE[cond], ax=ax)
fig.suptitle('ISI distributions — most active electrode per condition', fontsize=10)
fig.tight_layout()
plt.show()

### 6.4 STTC synchrony matrix

The Spike Time Tiling Coefficient (STTC; Cutts & Eglen 2014) measures
pairwise synchrony between electrodes, corrected for firing rate.
Values range from −1 (anti-correlated) to +1 (perfectly synchronous).
The diagonal is always 1.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
from py_mea_axion.viz.network_plots import plot_sttc_matrix

for ax, (well, cond) in zip(axes, [('A1','SCRM'), ('A2','KD4'), ('A3','KD5')]):
    plot_sttc_matrix(exp.sttc_matrices[well],
                     title=f"{well}  [{cond}]", ax=ax)
fig.suptitle('STTC pairwise synchrony matrices · DIV 28', fontsize=11)
fig.tight_layout()
plt.show()

### 6.5 Network burst timelines

Each row shows when network bursts occurred across the full recording
duration. A network burst requires ≥ 35 % of active electrodes to
burst simultaneously.

In [ ]:
from py_mea_axion.viz.network_plots import plot_network_burst_timeline

fig, axes = plt.subplots(12, 1, figsize=(12, 14), sharex=True)
for i, well in enumerate(ACTIVE_WELLS):
    cond = COND[well]
    nbs  = exp.network_bursts.get(well, [])
    plot_network_burst_timeline(
        nbs, exp.total_time_s,
        color=PALETTE[cond],
        title=f"{well} [{cond}]  n={len(nbs)}",
        ax=axes[i],
    )
    axes[i].set_xlabel('')
axes[-1].set_xlabel('Time (s)', fontsize=9)
fig.suptitle('Network burst timelines · Plate 2 · DIV 28', fontsize=12)
fig.tight_layout()
plt.show()

### 6.6 Condition comparison

Box plots with individual well data points for four key metrics.
Statistical testing is left to the user — see Section 7.

In [ ]:
METRICS = [
    ('mean_mfr_active_hz',    'Mean MFR\n(active electrodes, Hz)'),
    ('burst_rate_hz',         'Burst rate\n(bursts / electrode / s)'),
    ('mean_burst_duration_s', 'Mean burst\nduration (s)'),
    ('mean_sttc',             'Mean STTC'),
]

js         = exp.joined_summary()
js         = js[js['condition'].notna()]
conditions = ['SCRM', 'KD4', 'KD5']
rng        = np.random.default_rng(0)

fig, axes = plt.subplots(1, 4, figsize=(14, 4.5))
for ax, (metric, ylabel) in zip(axes, METRICS):
    for xi, cond in enumerate(conditions):
        vals = js.loc[js['condition'] == cond, metric].dropna().values
        col  = PALETTE[cond]
        ax.boxplot(vals, positions=[xi], widths=0.45,
                   patch_artist=True, showfliers=False,
                   boxprops=dict(facecolor=col, alpha=0.30),
                   medianprops=dict(color=col, linewidth=2.5),
                   whiskerprops=dict(color=col, linewidth=1.2),
                   capprops=dict(color=col, linewidth=1.2))
        ax.scatter(xi + rng.uniform(-0.13, 0.13, len(vals)), vals,
                   color=col, s=40, zorder=4, alpha=0.9,
                   edgecolors='white', linewidths=0.5)
    ax.set_xticks(range(3))
    ax.set_xticklabels(conditions, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.tick_params(labelsize=8)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('Condition comparison · Plate 2 · DIV 28  (n = 4 wells each)\n'
             'min_spikes=50 · max_ISI=0.1 s · participation ≥ 35 %',
             fontsize=10, y=1.03)
fig.tight_layout()
plt.show()

## 7. Statistical comparison

`compare_conditions` automatically selects Mann-Whitney U (two groups)
or Kruskal-Wallis + Dunn post-hoc (three or more groups).
Effect size is rank-biserial *r* (two groups) or ε² (≥ 3 groups).

In [ ]:
from py_mea_axion.stats.compare import compare_conditions

for metric in ['mean_mfr_active_hz', 'burst_rate_hz',
               'mean_burst_duration_s', 'mean_sttc']:
    res = compare_conditions(js, metric=metric, group_col='condition')
    print(f"\n{metric}")
    print(f"  test={res.test}  H={res.statistic:.3f}  p={res.p_value:.4f}  ε²={res.effect_size:.3f}")
    if res.posthoc is not None:
        print(res.posthoc[['group1','group2','z_stat','p_adjusted']].to_string(index=False))

## 8. Saving results

All result tables can be exported to CSV for downstream analysis in R,
GraphPad Prism, or any other tool.

In [ ]:
out = Path('notebook_output')
out.mkdir(exist_ok=True)

exp.spike_metrics.to_csv(out / 'spike_metrics.csv', index=False)
exp.burst_table.to_csv(out   / 'burst_table.csv',   index=False)
js.to_csv(out              / 'well_summary.csv',   index=False)

print('Saved:')
for f in sorted(out.glob('*.csv')):
    print(f'  {f.name}  ({f.stat().st_size / 1024:.1f} KB)')

## 9. Command-line equivalent

The same analysis can be run from the terminal without any Python:

```bash
mea-axion run \
    "../LGI2 KD data/20251004_LGI2 KD_Plate 2_D28N(000).spk" \
    --metadata plate2_metadata.csv \
    --fs-override 12500 \
    --min-spikes 50 \
    --max-isi 0.1 \
    --out results/
```

This writes `spike_metrics.csv`, `burst_table.csv`, `well_summary.csv`,
and a `figures/` folder with one set of plots per well.

---

## References

Cutts, C. S. & Eglen, S. J. (2014). Detecting pairwise correlations in
spike trains: an objective comparison of methods and application to the
study of retinal waves. *Journal of Neuroscience*, 34(43), 14288–14303.
https://doi.org/10.1523/JNEUROSCI.2767-14.2014

---

**Package repository:** https://github.com/Molecularbiologyworld/py-mea-axion  
**Archived release:** https://doi.org/10.5281/zenodo.19350375  
**License:** MIT